# 🔍 BHR Real-Time Sliding Window Detection

**Key Features:**
- ⏱️ **Sliding window** prediction (no future data leak)
- 🎯 Per-window BHR identification
- 🗺️ **Heatmap** visualization (routers × time)
- 📊 Real-time alert tracking

In [ ]:
#@title 1️⃣ Setup
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")

FEATURE_COLUMNS = [
    'flit_in', 'flit_out', 'avg_wait', 'max_wait', 'buffer_occ', 
    'active_vcs', 'stalls', 'credits', 'crossbar', 'io_ratio',
    'sw_in_arb', 'sw_out_arb', 'empty_vcs', 'total_wait', 
    'min_cred', 'max_cred', 'credit_sends'
]

class BHRAutoencoder(nn.Module):
    def __init__(self, input_dim=17, latent_dim=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 12), nn.ReLU(), nn.BatchNorm1d(12),
            nn.Linear(12, 8), nn.ReLU(), nn.BatchNorm1d(8),
            nn.Linear(8, latent_dim), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8), nn.ReLU(), nn.BatchNorm1d(8),
            nn.Linear(8, 12), nn.ReLU(), nn.BatchNorm1d(12),
            nn.Linear(12, input_dim)
        )
    def forward(self, x): return self.decoder(self.encoder(x))
    def get_error(self, x):
        with torch.no_grad(): return torch.mean((x - self.forward(x)) ** 2, dim=1)

print("✅ Setup complete")

In [ ]:
#@title 2️⃣ Upload & Train on Normal Data
print("📤 Upload anomaly_features_p0.csv (normal traffic)")
uploaded = files.upload()
df_train = pd.read_csv(list(uploaded.keys())[0]).replace([np.inf, -np.inf], np.nan).dropna()
print(f"✅ Loaded {len(df_train)} samples")

X = df_train[FEATURE_COLUMNS].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_val = int(len(X_scaled) * 0.2)
idx = np.random.permutation(len(X_scaled))
X_train = torch.FloatTensor(X_scaled[idx[n_val:]]).to(device)
X_val = torch.FloatTensor(X_scaled[idx[:n_val]]).to(device)

# Train
model = BHRAutoencoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

print("🏋️ Training...")
best_loss, best_state = float('inf'), None
for epoch in range(100):
    model.train()
    for i in range(0, len(X_train), 256):
        optimizer.zero_grad()
        loss = criterion(model(X_train[i:i+256]), X_train[i:i+256])
        loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad(): val_loss = criterion(model(X_val), X_val).item()
    if val_loss < best_loss: best_loss, best_state = val_loss, model.state_dict().copy()
    if (epoch+1) % 25 == 0: print(f"   Epoch {epoch+1}: Val={val_loss:.6f}")

model.load_state_dict(best_state)
errors = model.get_error(X_train).cpu().numpy()
threshold = np.mean(errors) + 3 * np.std(errors)
print(f"\n✅ Trained! Threshold: {threshold:.6f}")

In [ ]:
#@title 3️⃣ Upload Test Data (Attack Traffic)
print("📤 Upload attack data (p0.1 or p0.03)")
uploaded_test = files.upload()
df_test = pd.read_csv(list(uploaded_test.keys())[0]).replace([np.inf, -np.inf], np.nan).dropna()
print(f"✅ Loaded {len(df_test)} samples")

# Get all unique ticks and routers
unique_ticks = sorted(df_test['tick'].unique())
unique_routers = sorted(df_test['router_id'].unique())
print(f"   Time windows: {len(unique_ticks)}, Routers: {len(unique_routers)}")

In [ ]:
#@title 4️⃣ ⏱️ Sliding Window Detection (No Future Data!)
# Process each time window sequentially
window_results = []
cumulative_anomalies = {r: 0 for r in unique_routers}
cumulative_samples = {r: 0 for r in unique_routers}

print("🔍 Processing time windows...")
model.eval()

for i, tick in enumerate(unique_ticks):
    # Get data for this time window only
    window_df = df_test[df_test['tick'] == tick]
    
    # Predict anomalies for this window
    X_window = scaler.transform(window_df[FEATURE_COLUMNS].values)
    scores = model.get_error(torch.FloatTensor(X_window).to(device)).cpu().numpy()
    is_anomaly = scores > threshold
    
    # Per-router results for this window
    window_router_scores = {}
    for j, (_, row) in enumerate(window_df.iterrows()):
        rid = row['router_id']
        window_router_scores[rid] = scores[j]
        cumulative_anomalies[rid] += int(is_anomaly[j])
        cumulative_samples[rid] += 1
    
    # Identify suspected BHR using ONLY data up to this point
    rates = {r: cumulative_anomalies[r]/max(1,cumulative_samples[r]) for r in unique_routers}
    suspected_bhr = max(rates, key=rates.get)
    suspected_rate = rates[suspected_bhr]
    
    window_results.append({
        'tick': tick,
        'window_idx': i,
        'suspected_bhr': suspected_bhr if suspected_rate > 0.3 else None,
        'confidence': suspected_rate,
        'router_scores': window_router_scores.copy()
    })
    
    if (i+1) % 1000 == 0:
        print(f"   Window {i+1}/{len(unique_ticks)} - Current BHR: Router {suspected_bhr} ({suspected_rate:.1%})")

print(f"\n✅ Processed {len(window_results)} windows")

In [ ]:
#@title 5️⃣ 🗺️ Router x Time Heatmap
# Build heatmap matrix: rows=routers, cols=time windows
# Sample every Nth window for visualization
sample_rate = max(1, len(window_results) // 100)  # Max 100 columns
sampled_windows = window_results[::sample_rate]

heatmap_data = np.zeros((len(unique_routers), len(sampled_windows)))
for j, w in enumerate(sampled_windows):
    for i, rid in enumerate(unique_routers):
        if rid in w['router_scores']:
            heatmap_data[i, j] = w['router_scores'][rid]

# Create heatmap
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(heatmap_data, 
            xticklabels=[f"{w['tick']//1000000}M" for w in sampled_windows[::len(sampled_windows)//10+1]],
            yticklabels=unique_routers,
            cmap='YlOrRd',
            ax=ax,
            cbar_kws={'label': 'Anomaly Score'})

ax.axhline(y=unique_routers.index(10)+0.5, color='blue', linewidth=2, linestyle='--')
ax.axhline(y=unique_routers.index(10)+1.5, color='blue', linewidth=2, linestyle='--')
ax.set_title('🗺️ Router × Time Anomaly Heatmap', fontsize=14)
ax.set_xlabel('Time (tick)')
ax.set_ylabel('Router ID')

plt.tight_layout()
plt.savefig('heatmap.png', dpi=150)
plt.show()
print("📸 Saved: heatmap.png")

In [ ]:
#@title 6️⃣ 📊 BHR Detection Over Time
# Track when BHR was first detected and confidence over time
detection_timeline = []
first_detection = None

for w in window_results:
    if w['suspected_bhr'] is not None:
        if first_detection is None:
            first_detection = w
        detection_timeline.append({
            'tick': w['tick'],
            'bhr': w['suspected_bhr'],
            'confidence': w['confidence']
        })

print("🎯 BHR DETECTION TIMELINE")
print("="*60)
if first_detection:
    print(f"\n🚨 FIRST DETECTION:")
    print(f"   Router: {first_detection['suspected_bhr']}")
    print(f"   Time: {first_detection['tick']} ticks (window {first_detection['window_idx']})")
    print(f"   Confidence: {first_detection['confidence']:.1%}")
else:
    print("\n✅ No BHR detected")

# Plot confidence over time
if detection_timeline:
    df_timeline = pd.DataFrame(detection_timeline)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Confidence over time
    ax1.plot(df_timeline['tick'], df_timeline['confidence'], 'r-', linewidth=1)
    ax1.fill_between(df_timeline['tick'], 0, df_timeline['confidence'], alpha=0.3, color='red')
    ax1.axhline(0.5, color='gray', linestyle='--', label='50% confidence')
    ax1.set_ylabel('Confidence')
    ax1.set_title('📈 BHR Detection Confidence Over Time')
    ax1.legend()
    
    # Which router detected as BHR
    ax2.scatter(df_timeline['tick'], df_timeline['bhr'], c='red', s=10, alpha=0.5)
    ax2.set_ylabel('Suspected Router')
    ax2.set_xlabel('Time (tick)')
    ax2.set_yticks(unique_routers)
    ax2.set_title('🎯 Suspected BHR Router Over Time')
    
    plt.tight_layout()
    plt.savefig('detection_timeline.png', dpi=150)
    plt.show()

In [ ]:
#@title 7️⃣ 📋 Final Per-Window Summary
# Show detection at key time points
print("\n" + "="*70)
print("  ⏱️ SLIDING WINDOW DETECTION SUMMARY")
print("="*70)
print(f"{'Window':>8} {'Tick':>15} {'Suspected BHR':>15} {'Confidence':>12}")
print("-"*70)

# Show every 10% of windows
for pct in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    idx = min(int(len(window_results) * pct) - 1, len(window_results) - 1)
    w = window_results[idx]
    bhr_str = str(w['suspected_bhr']) if w['suspected_bhr'] else "-"
    print(f"{w['window_idx']:>8} {w['tick']:>15} {bhr_str:>15} {w['confidence']:>11.1%}")

# Final result
final = window_results[-1]
print("\n" + "="*70)
if final['suspected_bhr']:
    print(f"🚨 FINAL VERDICT: Router {final['suspected_bhr']} is BHR (Confidence: {final['confidence']:.1%})")
else:
    print("✅ No BHR detected")
print("="*70)

In [ ]:
#@title 8️⃣ 💾 Save & Download
# Save model
torch.save({'model_state': model.state_dict(), 'scaler': scaler, 'threshold': threshold}, 
           'bhr_autoencoder.pth')

# Save window results
results_df = pd.DataFrame([{
    'tick': w['tick'],
    'window_idx': w['window_idx'],
    'suspected_bhr': w['suspected_bhr'],
    'confidence': w['confidence']
} for w in window_results])
results_df.to_csv('sliding_window_results.csv', index=False)

print("📥 Downloading...")
files.download('bhr_autoencoder.pth')
files.download('heatmap.png')
files.download('detection_timeline.png')
files.download('sliding_window_results.csv')
print("\n✅ Done!")